# Perfiles de usuarios — tienda multicategoría Open CDP

Cuenta **SalmiBecerra** (`323884514768`). Región `us-east-2`.

Bucket: `proyecto-final-aws-glue-sagemaker-workshop-ecomerse-11-2026`.

Cada fila de origen es un evento entre un usuario y un producto (vista, carrito, quitar del carrito o compra). Este notebook segmenta usuarios con K-Means a partir de `gold_dim_usuario`.

## Paso 3 — Dependencias

Reinicie el kernel cuando la celda lo pida y continúe debajo.

In [ ]:
# SageMaker SDK v2 compatible con el taller
!pip uninstall -y sagemaker-train sagemaker-serve sagemaker-mlops -q
!pip install "sagemaker==2.232.2" "joblib<1.6" awswrangler scikit-learn matplotlib seaborn -q
!pip install --force-reinstall --no-deps "sagemaker==2.232.2" -q

import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)


In [ ]:
import sagemaker, awswrangler as wr, sklearn, joblib
print(sagemaker.__version__, joblib.__version__, sklearn.__version__)


## Configuración y lectura desde Athena

In [ ]:
import boto3
import awswrangler as wr
import pandas as pd
import numpy as np

BUCKET = 'proyecto-final-aws-glue-sagemaker-workshop-ecomerse-11-2026'
DATABASE = 'db_ecommerce'
REGION = 'us-east-2'
S3_OUT = f's3://{BUCKET}/athena-results/'
session = boto3.Session(region_name=REGION)

QUERY = """
SELECT
    user_id,
    total_vistas,
    total_carritos,
    total_remociones,
    total_compras,
    COALESCE(ticket_promedio, 0) AS ticket_promedio,
    sesiones,
    categorias_distintas,
    tasa_conversion,
    categoria_principal
FROM db_ecommerce.gold_dim_usuario
WHERE total_eventos > 0
  AND total_vistas IS NOT NULL
ORDER BY RAND()
LIMIT 100000
"""

df = wr.athena.read_sql_query(
    sql=QUERY,
    database=DATABASE,
    ctas_approach=True,
    s3_output=S3_OUT,
    boto3_session=session,
)

print(f'Registros cargados: {df.shape[0]:,}')
print(f'Columnas: {list(df.columns)}')
df.head()


## Paso 4 — Preparar datos para K-Means

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

FEATURES = [
    'total_vistas',
    'total_carritos',
    'total_remociones',
    'total_compras',
    'ticket_promedio',
    'sesiones',
    'categorias_distintas',
    'tasa_conversion',
    'categoria_principal',
]
CATEGORICAS = ['categoria_principal']

df_model = df[FEATURES].copy()
df_model['categoria_principal'] = df_model['categoria_principal'].fillna('sin_categoria').astype(str)
df_model['ticket_promedio'] = df_model['ticket_promedio'].fillna(0)
df_model = df_model.dropna()

encoders = {}
for col in CATEGORICAS:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col].astype(str))
    encoders[col] = le

scaler = StandardScaler()
X = scaler.fit_transform(df_model)
print(f'Matriz lista para K-Means: {X.shape}')


## Paso 5 — Elegir el número de clusters

In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

inertias = []
K_RANGE = range(2, 10)

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(list(K_RANGE), inertias, marker='o', color='steelblue', linewidth=2)
plt.xlabel('Numero de clusters (k)')
plt.ylabel('Inercia')
plt.title('Metodo del codo — perfiles de usuarios e-commerce')
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('elbow_curve.png', dpi=150)
plt.show()


## Paso 6 — Entrenar K-Means

In [ ]:
K_OPTIMO = 4  # ajuste segun la grafica del codo

km_final = KMeans(
    n_clusters=K_OPTIMO,
    random_state=42,
    n_init=10,
    max_iter=300,
)

df_model['cluster'] = km_final.fit_predict(X)
print('Usuarios por cluster:')
print(df_model['cluster'].value_counts().sort_index())


## Paso 7 — Interpretar perfiles

In [ ]:
df_resultado = df.loc[df_model.index, FEATURES].copy()
df_resultado['cluster'] = df_model['cluster'].values
df_resultado['user_id'] = df.loc[df_model.index, 'user_id'].values

perfil = df_resultado.groupby('cluster').agg(
    usuarios=('user_id', 'count'),
    vistas_promedio=('total_vistas', 'mean'),
    carritos_promedio=('total_carritos', 'mean'),
    compras_promedio=('total_compras', 'mean'),
    ticket_promedio=('ticket_promedio', 'mean'),
    tasa_conversion=('tasa_conversion', 'mean'),
    categoria_frecuente=('categoria_principal', lambda x: x.value_counts().index[0]),
).round(2)

print(perfil)

nombres_cluster = {
    0: 'Exploradores',
    1: 'Compradores frecuentes',
    2: 'Abandonan el carrito',
    3: 'Alto valor',
}
df_resultado['perfil'] = df_resultado['cluster'].map(nombres_cluster)


## Paso 8 — Guardar el modelo en S3

In [ ]:
import joblib, boto3, tarfile

S3_MODELOS = f's3://{BUCKET}/modelos/kmeans/'

joblib.dump(km_final, 'kmeans_model.joblib')
joblib.dump(scaler, 'scaler.joblib')
joblib.dump(encoders, 'encoders.joblib')

with tarfile.open('model.tar.gz', 'w:gz') as tar:
    tar.add('kmeans_model.joblib')
    tar.add('scaler.joblib')
    tar.add('encoders.joblib')

s3 = boto3.client('s3', region_name=REGION)
s3.upload_file('model.tar.gz', BUCKET, 'modelos/kmeans/model.tar.gz')

wr.s3.to_csv(
    df=df_resultado,
    path=f'{S3_MODELOS}resultados/kmeans_perfiles.csv',
    index=False,
)
print('Modelo y resultados guardados en S3')


## Paso 9 — Script de inferencia

In [ ]:
%%writefile inference.py
import joblib
import json
import numpy as np
import os

PERFILES = {
    0: {'nombre': 'Exploradores', 'descripcion': 'Muchas vistas y poca compra'},
    1: {'nombre': 'Compradores frecuentes', 'descripcion': 'Compras repetidas y conversion alta'},
    2: {'nombre': 'Abandonan el carrito', 'descripcion': 'Agregan productos al carrito y no compran'},
    3: {'nombre': 'Alto valor', 'descripcion': 'Pocas compras con ticket promedio alto'},
}

def model_fn(model_dir):
    model = joblib.load(os.path.join(model_dir, 'kmeans_model.joblib'))
    scaler = joblib.load(os.path.join(model_dir, 'scaler.joblib'))
    return {'model': model, 'scaler': scaler}

def input_fn(request_body, content_type):
    if isinstance(request_body, bytes):
        request_body = request_body.decode('utf-8')
    body = json.loads(request_body)
    if 'instancias' in body:
        data = body['instancias']
    elif 'instances' in body:
        data = body['instances']
    else:
        data = body
    return np.array(data, dtype=float)

def predict_fn(input_data, artifacts):
    X_scaled = artifacts['scaler'].transform(input_data)
    clusters = artifacts['model'].predict(X_scaled)
    return clusters.tolist()

def output_fn(prediction, accept):
    resultado = []
    for cluster in prediction:
        perfil = PERFILES.get(cluster, {'nombre': 'Desconocido', 'descripcion': 'Sin clasificacion'})
        resultado.append({
            'cluster': cluster,
            'perfil': perfil['nombre'],
            'descripcion': perfil['descripcion'],
        })
    return json.dumps({'predicciones': resultado})


## Paso 10 — Desplegar el endpoint

El endpoint `kmeans-perfiles-ecommerce` cobra por hora. Bórrelo al terminar el taller.

In [ ]:
import time
import boto3
import sagemaker
from sagemaker.sklearn.model import SKLearnModel

sm_client = boto3.client('sagemaker', region_name=REGION)

try:
    sm_client.delete_endpoint(EndpointName='kmeans-perfiles-ecommerce')
except Exception:
    print('Endpoint no existia')

try:
    sm_client.delete_endpoint_config(EndpointConfigName='kmeans-perfiles-ecommerce')
except Exception:
    print('Endpoint config no existia')

time.sleep(10)

ROLE = sagemaker.get_execution_role()
sklearn_model = SKLearnModel(
    model_data=f's3://{BUCKET}/modelos/kmeans/model.tar.gz',
    role=ROLE,
    entry_point='inference.py',
    framework_version='1.2-1',
    sagemaker_session=sagemaker.Session(),
)

predictor = sklearn_model.deploy(
    instance_type='ml.t2.medium',
    initial_instance_count=1,
    endpoint_name='kmeans-perfiles-ecommerce',
)
print(f'Endpoint activo: {predictor.endpoint_name}')


## Paso 11 — Probar el endpoint

El vector ya debe estar codificado y en el orden de `FEATURES`. El endpoint vuelve a escalar.

In [ ]:
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

predictor.serializer = JSONSerializer()
predictor.deserializer = JSONDeserializer()

payload = {'instancias': [
    [80, 1, 0, 0, 0.0, 6, 3, 0.0, 0],
    [20, 8, 1, 6, 45.0, 4, 2, 0.3, 1],
    [15, 9, 2, 0, 0.0, 3, 2, 0.0, 2],
    [4, 1, 0, 1, 420.0, 1, 1, 0.25, 0],
]}

response = predictor.predict(payload)
print('Predicciones:', response)


## Limpieza

```python
sm_client.delete_endpoint(EndpointName='kmeans-perfiles-ecommerce')
sm_client.delete_endpoint_config(EndpointConfigName='kmeans-perfiles-ecommerce')
```